1) Import everything we need

In [116]:

# Modules for model workflows and transformer building
import numpy as np
import random
import torch
from torch import nn
from torch.nn import TransformerEncoder, TransformerEncoderLayer, TransformerDecoderLayer, TransformerDecoder
from torch.utils.data import DataLoader
from torch.utils.data.dataset import TensorDataset
from torch.optim import Adam
from math import floor

# Modules for data generation
from scipy.signal import cont2discrete

# Modules for some custom loss function
from torch.linalg import inv

2) Setup seed for reproducibility

In [117]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

3) Define transformer class

In [118]:
class TransformerAutoencoder(nn.Module):
    def __init__(self, 
                 encoder_input_dim, 
                 decoder_input_dim, 
                 hidden_dim,
                 num_heads, 
                 encoder_embedding_dim, 
                 decoder_embedding_dim,
                 num_layers, 
                 dropout):
        super(TransformerAutoencoder, self).__init__()
        self.encoder_input_dim = encoder_input_dim
        self.decoder_input_dim = decoder_input_dim
        self.encoder_embedding_dim = encoder_embedding_dim
        self.decoder_embedding_dim = decoder_embedding_dim
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.dropout = dropout

        # Encoder Embedding
        self.encoder_embedding = nn.Linear(self.encoder_input_dim, 
                                           self.encoder_embedding_dim)

        # Encoder
        self.encoder_layer = TransformerEncoderLayer(d_model=self.encoder_embedding_dim,
                                                     nhead=self.num_heads,
                                                     dim_feedforward=self.hidden_dim,
                                                     dropout=self.dropout,
                                                     batch_first=True)
        self.encoder = TransformerEncoder(self.encoder_layer,
                                          num_layers=self.num_layers)

        # Decoder Embedding
        self.decoder_embedding = nn.Linear(self.decoder_input_dim, 
                                           self.decoder_embedding_dim)

        # Decoder
        self.decoder_layer = TransformerDecoderLayer(d_model=self.decoder_embedding_dim,
                                                     nhead=self.num_heads,
                                                     dim_feedforward=self.hidden_dim,
                                                     dropout=self.dropout,
                                                     batch_first=True)
        self.decoder = TransformerDecoder(self.decoder_layer, 
                                          num_layers=self.num_layers)

        # Final output layer
        self.out = nn.Linear(self.decoder_embedding_dim,
                             self.decoder_input_dim)

    def forward(self, inputs, targets):        
        # Encode the input
        encoded_input = self.encoder(self.encoder_embedding(inputs))
        
        # Decode the target
        decoder_input = self.decoder_embedding(targets)
        decoded_target = self.decoder(decoder_input, encoded_input)        
        
        # Apply the final output layer
        target_output = self.out(decoded_target)
        return target_output

4) Data for our experiments: Lorrentz data, Linear springs data, Kuramoto Shivasinsky

a) Lorrentz Data: 
n = num of sequence
allInputs = Y_data = measurements [nx1]
allTargets = X_data = states [nx3]

In [119]:
# rawData = np.load("../Data/lorenz_data_hackathon.npz")
# allInputs = torch.Tensor(rawData["Y_data"])
# allTargets = torch.Tensor(rawData["X_data"])
# numSequence = allInputs.shape[0]
# encoder_input_dim = allInputs.shape[2]
# decoder_input_dim = allTargets.shape[2]

b) Linear springs data:
n = num of sequence
allInputs = Y_data = measurements [nx1]
allTargets = X_data = states [nx3]

In [120]:
def data_generation(num_sequences, sequence_length, number_masses):
    dim_y = number_masses
    dim_x= 2*dim_y 

    X_data_array = np.empty((num_sequences, sequence_length, dim_x))
    Y_data_array = np.empty((num_sequences, sequence_length, dim_y))

    m = np.ones(dim_y)
    m = 10*m
   
    k = np.ones(dim_y)
    k = 800*k

    d = np.ones(dim_y)
    d = 6*d

    A_c = np.zeros((dim_x,dim_x))

    offset = 0
    for i in range(dim_x):
        if i % 2 == 0:
            A_c[i,i+1] = 1

        if i % 2 == 1:
            if i != dim_x-1:
                A_c[i,i-1] = -(k[i-1-offset]+k[i-offset])/m[i-1-offset]
                A_c[i,i] = -(d[i-1-offset]+d[i-offset])/m[i-1-offset]
                A_c[i,i+1] = k[i-offset]/m[i-1-offset]
                A_c[i,i+2] = d[i-offset]/m[i-1-offset]
            else:
                A_c[i,i-1] = -k[i-dim_y]/m[i-dim_y]
                A_c[i,i] = -d[i-dim_y]/m[i-dim_y]

            if i != 1:
                A_c[i,i-3] = k[i-1-offset]/m[i-1-offset]
                A_c[i,i-2] = d[i-1-offset]/m[i-1-offset]

            offset += 1

    B_c = np.zeros((dim_x,dim_y))

    H_c = np.zeros((dim_y,dim_x))
    offset = 0
    for i in range(dim_y):
        H_c[i,i+offset] = 1

        offset += 1

    D_c = np.array([[0.]])

   
    dt = 0.1 
    d_system = cont2discrete((A_c, B_c, H_c, D_c),dt)
    A = d_system[0] 
    H = d_system[2] 

    def is_schur(matrix):
       
        eigenvalues, _ = np.linalg.eig(matrix)
        if np.all(np.abs(eigenvalues) < 1):
            print(np.abs(eigenvalues))
            return True
        else:
            return False

    # if is_schur(A):
    #     print("The matrix is Schur.")
    # else:
    #     print("The matrix is not Schur.")


    sigma_p = 0.01 
    sigma_p_diag = (sigma_p**2)*np.ones(dim_x)
    Q = np.diag(sigma_p_diag)

    sigma_m = 0.01 
    sigma_m_diag = (sigma_m**2)*np.ones(dim_y)
    R = np.diag(sigma_m_diag)

    sigma_x = 0.01 
    sigma_x_diag = (sigma_p**2)*np.ones(dim_x)
    P = np.diag(sigma_x_diag)


    for s in range(num_sequences):
        mu_x0 = np.random.uniform(-10,10,size=dim_x) 
        x=np.random.multivariate_normal(mu_x0,P)
       
        X_data_array[s,0,:] = np.squeeze(np.asarray(x))

       
        v_0 = np.random.multivariate_normal(np.zeros(dim_y),R).reshape(-1,1)

        y = H.dot(x.reshape(-1,1)) + v_0


        W = np.random.multivariate_normal(np.zeros(dim_x), Q, sequence_length)
        V = np.random.multivariate_normal(np.zeros(dim_y), R, sequence_length)
        Y_data_array[s,0,:] = np.squeeze(np.asarray(y))

        for t in range(1,sequence_length+1):
            w = W[t-1].reshape(-1,1) 
            v = V[t-1].reshape(-1,1)
        
            x = A.dot(x.reshape(-1,1)) + w 
            y = H.dot(x.reshape(-1,1)) + v 

            X_data_array[s,t:t+1,:] = x.T
            Y_data_array[s,t:t+1,:] = y.T

    return X_data_array, Y_data_array, A, H, Q, R, P

X, Y, A, H, Q, R, P = data_generation(num_sequences=500, sequence_length = 100, number_masses = 10)
H = torch.Tensor(H)
Q = torch.Tensor(Q)
R = torch.Tensor(R)
P = torch.Tensor(P)
def f(x): return A @ x
def h(x): return H @ x



allInputs = torch.Tensor(Y)
allTargets = torch.Tensor(X)

numSequence = allInputs.shape[0]
encoder_input_dim = allInputs.shape[2]
decoder_input_dim = allTargets.shape[2]

5) Split the data into training, validation and testing

In [121]:
trainPercent = 0.7
testPercent = 0.15
validatePercent = 0.15
lenTrainingData = floor(trainPercent*numSequence)
lenTestingData = floor(testPercent*numSequence)
lenValidatingData = numSequence-lenTestingData-lenTrainingData

print("## Split the data:\n")
print(f"Training data set size   : {lenTrainingData}\n",
      f"Validation data set size : {lenTestingData}\n",
      f"Test data set size       : {lenValidatingData}")

trainingTensorDataSet = TensorDataset(allInputs[0:lenTestingData-1],
                                      allTargets[0:lenTestingData-1])
testingTensorDataSet = TensorDataset(allInputs[lenTestingData:lenTestingData+lenTrainingData-1],
                                     allTargets[lenTestingData:lenTestingData+lenTrainingData-1])
validatingTensorDataSet = TensorDataset(allInputs[lenTestingData+lenTestingData:],
                                        allTargets[lenTestingData+lenTestingData:])

## Split the data:

Training data set size   : 350
 Validation data set size : 75
 Test data set size       : 75


6) Define MSE loss function evaluator. Do back propagation as we evaluate the loss for every dataset we train

In [122]:
def evaluateMSELoss(model, optimizer, train_dataloader):
    sum_loss = 0
    criterion = nn.MSELoss()
    for i, batch in enumerate(train_dataloader):
        inputs, targets = batch
        optimizer.zero_grad()
        outputs = model(inputs, targets)
        loss = criterion(outputs, targets)
        sum_loss+=loss.item() 
        loss.backward()
        optimizer.step()
    avg_loss = sum_loss/len(train_dataloader)
    return avg_loss

def evaluateCustomLoss1(model, optimizer, train_dataloader, alpha):
    sum_loss = 0
    criterion = nn.MSELoss()
    for i, batch in enumerate(train_dataloader):
        inputs, targets = batch
        optimizer.zero_grad()
        outputs = model(inputs, targets)
        h_of_outputs = 0.0*inputs
        loss = alpha * criterion(outputs, targets) + (1.0 - alpha)*criterion(inputs[0,:,:], h(outputs[0,:,:].T).T)
        sum_loss+=loss.item() 
        loss.backward()
        optimizer.step()
    avg_loss = sum_loss/len(train_dataloader)
    return avg_loss

7) Define network training and validating parameters

In [123]:
alpha = 0.6667
num_layers = 4
num_epochs = 10
hidden_dim = 64
num_heads = 16
encoder_embedding_dim = 32
decoder_embedding_dim = 32
learn_rate = 0.005
weight_decay = 0.05
dropout = 0.05

trainDataSetLoader =  DataLoader(trainingTensorDataSet, batch_size = 1)
validationDataSetLoader = DataLoader(validatingTensorDataSet)
testDataSetLoader = DataLoader(testingTensorDataSet)

# Data dict to store best results
best_result = dict({
    'model':[],
    'learn_rate':[],
    'num_epochs':[],
    'encoder_embedding_dim':[],
    'decoder_embedding_dim':[],
    'hidden_dim':[],
    'num_heads':[],
    'weight_decay':[],
    'dropout':[],
    'avg_training_loss':[],
    'best_validation_loss':[]
})

8) Define model and optimizer

In [124]:


model = TransformerAutoencoder(encoder_input_dim, 
                               decoder_input_dim,
                               hidden_dim, 
                               num_heads, 
                               encoder_embedding_dim,
                               decoder_embedding_dim, 
                               num_layers, 
                               dropout)
optimizer = Adam(model.parameters(), 
                 lr=learn_rate, 
                 weight_decay=weight_decay)
    


    


In [ ]:
# initialise the best and avg losses
best_validation_loss = 1000000.
avg_training_loss = 0.
avg_validation_loss = 0.

for epoch in range(num_epochs):
    print(f"EPOCH NUMBER: {epoch+1}")
    model.train(True)
    avg_training_loss = evaluateCustomLoss1(model, optimizer, trainDataSetLoader, alpha)
    model.eval()
    
    sum_validation_loss = 0.0

    with torch.no_grad():
        for i, batch in enumerate(validationDataSetLoader):
            inputs, targets = batch
            outputs = model(inputs, targets)
            criterion = nn.MSELoss()
            val_loss = criterion(outputs, targets)
            sum_validation_loss += val_loss.item()
    avg_validation_loss = sum_validation_loss / len(validationDataSetLoader)
    
    print(f"AVERAGE TRAINING LOSS  : {avg_training_loss}\nAVERAGE VALIDATION LOSS: {avg_validation_loss}")

    if avg_validation_loss < best_validation_loss:
        best_validation_loss = avg_validation_loss
        print(f"BEST VALIDATION LOSS: {best_validation_loss} at EPOCH {epoch+1}")
        best_result['model_state_dict']=model.state_dict()
        best_result['optimizer_state_dict']=optimizer.state_dict() 
        best_result['learn_rate']=learn_rate
        best_result['encoder_embedding_dim']=encoder_embedding_dim
        best_result['decoder_embedding_dim'] =decoder_embedding_dim
        best_result['num_epochs']=num_epochs
        best_result['hidden_dim']=hidden_dim
        best_result['num_heads']=num_heads
        best_result['weight_decay']=weight_decay
        best_result['dropout']=dropout
        best_result['avg_training_loss']=avg_training_loss
        best_result['best_validation_loss']=best_validation_loss




EPOCH NUMBER: 1
AVERAGE TRAINING LOSS  : 61.1427533433244
AVERAGE VALIDATION LOSS: 75.62723521096366
BEST VALIDATION LOSS: 75.62723521096366 at EPOCH 1
EPOCH NUMBER: 2
AVERAGE TRAINING LOSS  : 44.35207406894581
AVERAGE VALIDATION LOSS: 55.03579986027309
BEST VALIDATION LOSS: 55.03579986027309 at EPOCH 2
EPOCH NUMBER: 3
AVERAGE TRAINING LOSS  : 32.786017482345166
AVERAGE VALIDATION LOSS: 41.1998676776886
BEST VALIDATION LOSS: 41.1998676776886 at EPOCH 3
EPOCH NUMBER: 4
AVERAGE TRAINING LOSS  : 25.032312715375745
AVERAGE VALIDATION LOSS: 32.72831060545785
BEST VALIDATION LOSS: 32.72831060545785 at EPOCH 4
EPOCH NUMBER: 5
AVERAGE TRAINING LOSS  : 20.342188860919023
AVERAGE VALIDATION LOSS: 26.447730985369002
BEST VALIDATION LOSS: 26.447730985369002 at EPOCH 5
EPOCH NUMBER: 6


In [ ]:
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
}, 'linear_checkpoint_latest.pth')
checkpoint = torch.load('linear_checkpoint_latest.pth')
# Restore model & optimizer
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1  # Resume from next epoch
loss = checkpoint['loss']

/tmp/ipykernel_301229/3996086605.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('checkpoint_latest.pth')


9) Test the unseen data

In [ ]:
model.eval()
sum_testing_loss = 0.0
total_seqs = 0

criterion = nn.MSELoss()
for i, batch in enumerate(testDataSetLoader):
    inputs, targets = batch
    outputs = model(inputs, targets)
    criterion = nn.MSELoss()
    test_loss = criterion(outputs, targets)
    sum_testing_loss += test_loss.item()
avg_testing_loss = sum_testing_loss / len(testDataSetLoader)
print(f"[MSE] = {avg_testing_loss:.6f}")

[MSE] = 0.127890


In [ ]:
print(x_true.shape)

torch.Size([1, 100, 20])
